<!-- PROFESSIONAL_HEADER_v1 -->
<div align="center">

# 🚛 FleetLogix
## Avance 1 · Generación de Datos
### *Diseño del schema relacional + Faker para 535k registros sintéticos*

**Henry Data Science · Proyecto 2 · Dody Dueñas**  
📧 dodydurema67@gmail.com · 🗓 2026

</div>

---

## 🎯 Objetivos de aprendizaje

- Definir un schema PostgreSQL normalizado para una operación de logística
- Generar datos sintéticos realistas (es_CO) con Faker para 6 entidades
- Insertar ~535.000 filas con buenas prácticas (batches, transactions, COPY)
- Validar integridad referencial y distribuciones

## 📦 Entregables

- `sql/schema.sql` con DDL completo
- Datos cargados en PostgreSQL listos para análisis
- Métricas de carga (tiempo, throughput)

## 🧭 Cómo navegar este notebook

1. Ejecuta las celdas de **arriba hacia abajo** la primera vez.
2. Cada bloque tiene una celda markdown que explica el *por qué* antes del código.
3. Los outputs incluyen métricas, tiempos y validaciones.
4. Si interrumpes, todas las operaciones de BD usan transactions y son seguras de reintentar.

## 🔗 Pre-requisitos

- Haber corrido **`Avance_0_QuickStart.ipynb`** y verificado que todos los chequeos están en verde.
- PostgreSQL corriendo via `docker compose up -d` o el script `dashboard/setup/setup.ps1`.

---


<meta charset="UTF-8">

<div class="header">
<h1>🚛 FleetLogix Enterprise</h1>
<h2>Generación de Datos Masivos - Proyecto Integrador M2</h2>
<div class="info-bar">
<span>👤 <strong>Autor:</strong> Dody Dueñas</span>
<span>📅 <strong>Fecha:</strong> Abril 2026</span>
<span>🎓 <strong>Campus:</strong> Henry Data Science</span>
</div>
</div>

<div class="section">
<h2>📋 1. Resumen Ejecutivo</h2>
<p>Este notebook documenta la fase de <strong>generación de datos masivos</strong> para el ecosistema FleetLogix, un sistema de gestión de transporte y logística corporativo de nivel empresarial.</p>
<h3>🎯 Objetivos del Avance</h3>
<ul>
<li><span class="badge">✅</span> Población coherente de las 6 tablas del modelo relacional</li>
<li><span class="badge">✅</span> Generación de 500k+ registros realistas</li>
<li><span class="badge">✅</span> Respeto de integridad referencial</li>
<li><span class="badge">✅</span> Análisis del modelo relacional y diagrama ER</li>
<li><span class="badge">✅</span> Validación de calidad de datos (DQA)</li>
</ul>
<h3>📊 Resultados Esperados</h3>
<table>
<tr><th>Tabla</th><th>Registros</th><th>Descripción</th></tr>
<tr><td><strong>vehicles</strong></td><td>1,000</td><td>Catálogo de vehículos de la flota</td></tr>
<tr><td><strong>drivers</strong></td><td>5,000</td><td>Registro de conductores</td></tr>
<tr><td><strong>routes</strong></td><td>500</td><td>Rutas predefinidas</td></tr>
<tr><td><strong>trips</strong></td><td>130,000</td><td>Viajes realizados</td></tr>
<tr><td><strong>deliveries</strong></td><td>400,000+</td><td>Entregas individuales</td></tr>
<tr><td><strong>maintenance</strong></td><td>5,000</td><td>Historial de mantenimiento</td></tr>
</table>
</div>

<br><div align="center" style="background-color: #1a1a24; padding: 25px; border-radius: 12px; border: 1px solid #333; margin: 25px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
    <h2 style="color: #F8B400; font-family: 'Segoe UI', sans-serif; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 5px;">🗄️ Modelo de Entidad-Relación (ER)</h2>
    <p style="color: #bbb; font-size: 15px; margin-bottom: 25px; font-family: sans-serif;">Estructura relacional (3NF) gestionando el histórico de transacciones en PostgreSQL.</p>
    <img src="../docs/assets/er_diagram.png" style="max-width: 95%; border-radius: 8px; box-shadow: 0 5px 15px rgba(0,0,0,0.8); border: 1px solid #444;" />
</div><br>

<div class="section">
<h2>⚙️ 3. Configuración del Entorno</h2>
<p>Configuramos las librerías necesarias para la generación de datos:</p>
</div>

In [1]:
import os, sys, logging
from datetime import datetime, timedelta
from typing import List, Tuple, Dict, Any
import random

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from faker import Faker

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

print("Entorno configurado correctamente")
print(f"   - Pandas: {pd.__version__}")
print(f"   - NumPy: {np.__version__}")
print(f"   - Seaborn: {sns.__version__}")
print(f"   - Faker: {Faker.__module__}")

Entorno configurado correctamente
   - Pandas: 2.2.1
   - NumPy: 1.26.4
   - Seaborn: 0.13.2
   - Faker: faker.proxy


<div class="section">
<h2>🔌 4. Conexión a PostgreSQL</h2>
<p>Establecemos la conexión a la base de datos PostgreSQL:</p>
</div>

In [2]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

    DB_CONFIG = {'dbname': os.getenv('DB_NAME', 'fleetlogix_db'), 'user': os.getenv('DB_USER', 'postgres'), 'password': os.getenv('DB_PASSWORD', ''), 'host': os.getenv('DB_HOST', 'localhost'), 'port': int(os.getenv('DB_PORT', 5432))}

def get_db_connection():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        logging.info("Conexion a PostgreSQL establecida")
        return conn
    except Exception as e:
        logging.error(f"Error de conexion: {e}")
        raise

conn = get_db_connection()
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()
print("Conexion a PostgreSQL exitosa")
print(f"   - Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"   - Base de datos: {DB_CONFIG['dbname']}")

2026-04-18 11:57:19,032 - INFO - Conexion a PostgreSQL establecida


Conexion a PostgreSQL exitosa
   - Host: localhost:5432
   - Base de datos: fleetlogix_db


<div class="section">
<h2>🏗️ 5. Inicialización del Esquema</h2>
<p>Creamos el esquema de base de datos con las 6 tablas y sus relaciones:</p>
</div>

In [3]:
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;")
existing_tables = [row[0] for row in cur.fetchall()]
print(f"Tablas existentes: {existing_tables}")

if existing_tables:
    print("Limpiando esquema existente...")
    tables_to_drop = ['maintenance', 'deliveries', 'trips', 'routes', 'drivers', 'vehicles']
    for table in tables_to_drop:
        cur.execute(f"DROP TABLE IF EXISTS {table} CASCADE")
        print(f"   Eliminada tabla: {table}")

print("Creando esquema de base de datos...")
schema_sql = """
CREATE TABLE vehicles (
    vehicle_id SERIAL PRIMARY KEY,
    license_plate VARCHAR(20) UNIQUE NOT NULL,
    vehicle_type VARCHAR(50) NOT NULL,
    capacity_kg DECIMAL(10,2),
    fuel_type VARCHAR(20),
    acquisition_date DATE,
    status VARCHAR(20) DEFAULT 'active'
);
CREATE TABLE drivers (
    driver_id SERIAL PRIMARY KEY,
    employee_code VARCHAR(20) UNIQUE NOT NULL,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    license_number VARCHAR(50) UNIQUE NOT NULL,
    license_expiry DATE,
    phone VARCHAR(20),
    hire_date DATE,
    status VARCHAR(20) DEFAULT 'active'
);
CREATE TABLE routes (
    route_id SERIAL PRIMARY KEY,
    route_code VARCHAR(20) UNIQUE NOT NULL,
    origin_city VARCHAR(100) NOT NULL,
    destination_city VARCHAR(100) NOT NULL,
    distance_km DECIMAL(10,2),
    estimated_duration_hours DECIMAL(5,2),
    toll_cost DECIMAL(10,2) DEFAULT 0
);
CREATE TABLE trips (
    trip_id SERIAL PRIMARY KEY,
    vehicle_id INTEGER REFERENCES vehicles(vehicle_id),
    driver_id INTEGER REFERENCES drivers(driver_id),
    route_id INTEGER REFERENCES routes(route_id),
    departure_datetime TIMESTAMP NOT NULL,
    arrival_datetime TIMESTAMP,
    fuel_consumed_liters DECIMAL(10,2),
    total_weight_kg DECIMAL(10,2),
    status VARCHAR(20) DEFAULT 'in_progress'
);
CREATE TABLE deliveries (
    delivery_id SERIAL PRIMARY KEY,
    trip_id INTEGER REFERENCES trips(trip_id),
    tracking_number VARCHAR(50) UNIQUE NOT NULL,
    customer_name VARCHAR(200) NOT NULL,
    delivery_address TEXT NOT NULL,
    package_weight_kg DECIMAL(10,2),
    scheduled_datetime TIMESTAMP,
    delivered_datetime TIMESTAMP,
    delivery_status VARCHAR(20) DEFAULT 'pending',
    recipient_signature BOOLEAN DEFAULT FALSE
);
CREATE TABLE maintenance (
    maintenance_id SERIAL PRIMARY KEY,
    vehicle_id INTEGER REFERENCES vehicles(vehicle_id),
    maintenance_date DATE NOT NULL,
    maintenance_type VARCHAR(50) NOT NULL,
    description TEXT,
    cost DECIMAL(10,2),
    next_maintenance_date DATE,
    performed_by VARCHAR(200)
);
CREATE INDEX idx_trips_departure ON trips(departure_datetime);
CREATE INDEX idx_deliveries_status ON deliveries(delivery_status);
CREATE INDEX idx_vehicles_status ON vehicles(status);
COMMENT ON TABLE vehicles IS 'Registro de vehiculos de la flota';
COMMENT ON TABLE drivers IS 'Informacion de conductores empleados';
COMMENT ON TABLE routes IS 'Rutas predefinidas entre ciudades';
COMMENT ON TABLE trips IS 'Registro de viajes realizados';
COMMENT ON TABLE deliveries IS 'Entregas individuales asociadas a cada viaje';
COMMENT ON TABLE maintenance IS 'Historial de mantenimiento de vehiculos';
"""
cur.execute(schema_sql)
print("Esquema creado exitosamente")
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' AND table_type = 'BASE TABLE' ORDER BY table_name;")
tables_created = [row[0] for row in cur.fetchall()]
print(f"Tablas creadas: {tables_created}")

Tablas existentes: ['customers', 'deliveries', 'dim_date', 'drivers', 'maintenance', 'routes', 'system_logs', 'trips', 'v_dim_date', 'vehicles']
Limpiando esquema existente...
   Eliminada tabla: maintenance
   Eliminada tabla: deliveries
   Eliminada tabla: trips
   Eliminada tabla: routes
   Eliminada tabla: drivers
   Eliminada tabla: vehicles
Creando esquema de base de datos...
Esquema creado exitosamente
Tablas creadas: ['customers', 'deliveries', 'dim_date', 'drivers', 'maintenance', 'routes', 'system_logs', 'trips', 'vehicles']


<div class="section">
<h2>📊 6. Generación de Datos Masivos</h2>
</div>

In [4]:
fake = Faker('es_CO')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

VEHICLE_TYPES = ['Camion Grande', 'Camion Mediano', 'Van', 'Motocicleta']
FUEL_TYPES = ['diesel', 'gasolina', 'electrico', 'hibrido']
VEHICLE_CAPACITIES = {'Camion Grande': 5000, 'Camion Mediano': 3000, 'Van': 1500, 'Motocicleta': 50}
COLOMBIAN_CITIES = ['Bogota', 'Medellin', 'Cali', 'Barranquilla', 'Cartagena', 'Cucuta', 'Bucaramanga', 'Pereira', 'Manizales', 'Ibague', 'Neiva', 'Pasto', 'Monteria', 'Villavicencio', 'Santa Marta']
MAINTENANCE_TYPES = ['Cambio de aceite', 'Revision de frenos', 'Alineacion y balanceo', 'Cambio de neumaticos', 'Revision de motor', 'Sistema electrico', 'Cambio de filtros', 'Inspeccion general', 'Mantenimiento preventivo']
DELIVERY_STATUSES = ['pending', 'in_transit', 'delivered', 'failed', 'cancelled']
TRIP_STATUSES = ['in_progress', 'completed', 'cancelled']
VEHICLE_STATUSES = ['active', 'maintenance', 'inactive']

print("Configuracion de generacion cargada")
print(f"   - Tipos de vehiculo: {VEHICLE_TYPES}")
print(f"   - Ciudades: {len(COLOMBIAN_CITIES)}")
print(f"   - Tipos de mantenimiento: {len(MAINTENANCE_TYPES)}")

Configuracion de generacion cargada
   - Tipos de vehiculo: ['Camion Grande', 'Camion Mediano', 'Van', 'Motocicleta']
   - Ciudades: 15
   - Tipos de mantenimiento: 9


<div class="section">
<h3>6.1 🚛 Generación de Vehicles</h3>
</div>

In [5]:
def generate_license_plate():
    letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    return f"{random.choice(letters)}{random.choice(letters)}{random.choice(letters)}{random.randint(100, 999)}"

def generate_vehicles(count):
    vehicles = []
    used_plates = set()
    start_date = datetime(2020, 1, 1)
    end_date = datetime(2025, 12, 31)
    for _ in range(count):
        plate = generate_license_plate()
        while plate in used_plates:
            plate = generate_license_plate()
        used_plates.add(plate)
        vehicle_type = random.choice(VEHICLE_TYPES)
        capacity = VEHICLE_CAPACITIES[vehicle_type] * random.uniform(0.9, 1.1)
        fuel_type = random.choice(FUEL_TYPES)
        acquisition_date = fake.date_between(start_date=start_date, end_date=end_date)
        status = random.choices(VEHICLE_STATUSES, weights=[85, 10, 5])[0]
        vehicles.append((plate, vehicle_type, capacity, fuel_type, acquisition_date, status))
    return vehicles

print("Generando Vehicles...")
start_time = datetime.now()
vehicles_data = generate_vehicles(1000)
cur.executemany("INSERT INTO vehicles (license_plate, vehicle_type, capacity_kg, fuel_type, acquisition_date, status) VALUES (%s, %s, %s, %s, %s, %s)", vehicles_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(vehicles_data):,} vehiculos generados en {duration:.2f}s")
cur.execute("SELECT vehicle_id FROM vehicles ORDER BY vehicle_id")
vehicle_ids = [row[0] for row in cur.fetchall()]
print(f"   Rango de IDs: {min(vehicle_ids)} - {max(vehicle_ids)}")

Generando Vehicles...
1,000 vehiculos generados en 1.07s
   Rango de IDs: 1 - 1000


<div class="section">
<h3>6.2 👨‍💼 Generación de Drivers</h3>
</div>

In [6]:
def generate_drivers(count):
    drivers = []
    used_codes, used_licenses = set(), set()
    hire_start, hire_end = datetime(2018, 1, 1), datetime(2025, 6, 30)
    for i in range(count):
        emp_code = f"EMP{i:05d}"
        while emp_code in used_codes:
            i += 1
            emp_code = f"EMP{i:05d}"
        used_codes.add(emp_code)
        first_name, last_name = fake.first_name(), fake.last_name()
        license_num = f"{random.randint(100000000, 999999999)}"
        while license_num in used_licenses:
            license_num = f"{random.randint(100000000, 999999999)}"
        used_licenses.add(license_num)
        license_expiry = fake.date_between(start_date=datetime(2025, 1, 1), end_date=datetime(2028, 12, 31))
        phone = f"+57{random.randint(3000000000, 3999999999)}"
        hire_date = fake.date_between(start_date=hire_start, end_date=hire_end)
        status = random.choices(VEHICLE_STATUSES, weights=[90, 5, 5])[0]
        drivers.append((emp_code, first_name, last_name, license_num, license_expiry, phone, hire_date, status))
    return drivers

print("Generando Drivers...")
start_time = datetime.now()
drivers_data = generate_drivers(5000)
cur.executemany("INSERT INTO drivers (employee_code, first_name, last_name, license_number, license_expiry, phone, hire_date, status) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)", drivers_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(drivers_data):,} conductores generados en {duration:.2f}s")
cur.execute("SELECT driver_id FROM drivers ORDER BY driver_id")
driver_ids = [row[0] for row in cur.fetchall()]
print(f"   Rango de IDs: {min(driver_ids)} - {max(driver_ids)}")

Generando Drivers...
5,000 conductores generados en 2.57s
   Rango de IDs: 1 - 5000


<div class="section">
<h3>6.3 🗺️ Generación de Routes</h3>
</div>

In [7]:
def generate_routes(count):
    routes = []
    for i in range(count):
        route_code = f"R{i:03d}"
        origin = random.choice(COLOMBIAN_CITIES)
        destination = random.choice([c for c in COLOMBIAN_CITIES if c != origin])
        distance = random.uniform(50, 1200)
        duration = distance / 60 + random.uniform(-1, 2)
        toll = distance * random.uniform(100, 200)
        routes.append((route_code, origin, destination, round(distance, 2), round(duration, 2), round(toll, 2)))
    return routes

print("Generando Routes...")
start_time = datetime.now()
routes_data = generate_routes(500)
cur.executemany("INSERT INTO routes (route_code, origin_city, destination_city, distance_km, estimated_duration_hours, toll_cost) VALUES (%s, %s, %s, %s, %s, %s)", routes_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(routes_data):,} rutas generadas en {duration:.2f}s")
cur.execute("SELECT route_id FROM routes ORDER BY route_id")
route_ids = [row[0] for row in cur.fetchall()]
print(f"   Rango de IDs: {min(route_ids)} - {max(route_ids)}")

Generando Routes...
500 rutas generadas en 0.23s
   Rango de IDs: 1 - 500


<div class="section">
<h3>6.4 🚚 Generación de Trips (130,000 registros)</h3>
</div>

In [8]:
def generate_trips(count, vehicle_ids, driver_ids, route_ids):
    trips = []
    start_date = datetime(2025, 1, 1)
    base_datetime = start_date
    for _ in range(count):
        vehicle_id, driver_id, route_id = random.choice(vehicle_ids), random.choice(driver_ids), random.choice(route_ids)
        cur.execute("SELECT distance_km FROM routes WHERE route_id = %s", (route_id,))
        distance = float(cur.fetchone()[0])
        cur.execute("SELECT capacity_kg FROM vehicles WHERE vehicle_id = %s", (vehicle_id,))
        capacity = float(cur.fetchone()[0])
        departure = base_datetime + timedelta(days=random.randint(0, 365), hours=random.randint(0, 23))
        duration_hours = distance / 60 + random.uniform(-0.5, 1.5)
        arrival = departure + timedelta(hours=duration_hours)
        fuel_consumed = (distance / 100) * random.uniform(20, 30)
        weight = random.uniform(100, capacity * 0.9)
        status = random.choices(TRIP_STATUSES, weights=[10, 85, 5])[0]
        trips.append((vehicle_id, driver_id, route_id, departure, arrival if status == 'completed' else None, round(fuel_consumed, 2), round(weight, 2), status))
    return trips

print("Generando Trips (130,000 registros)...")
start_time = datetime.now()
trips_data = generate_trips(130000, vehicle_ids, driver_ids, route_ids)
cur.executemany("INSERT INTO trips (vehicle_id, driver_id, route_id, departure_datetime, arrival_datetime, fuel_consumed_liters, total_weight_kg, status) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)", trips_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(trips_data):,} viajes generados en {duration:.2f}s")
cur.execute("SELECT trip_id FROM trips ORDER BY trip_id")
trip_ids = [row[0] for row in cur.fetchall()]
print(f"   Rango de IDs: {min(trip_ids):,} - {max(trip_ids):,}")

Generando Trips (130,000 registros)...
130,000 viajes generados en 76.87s
   Rango de IDs: 1 - 130,000


<div class="section">
<h3>6.5 📦 Generación de Deliveries (400,000 registros)</h3>
</div>

In [9]:
# Contador global para tracking numbers unicos garantizados
_tracking_counter = 0
def generate_tracking_number():
    global _tracking_counter
    _tracking_counter += 1
    letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    return f"FL{_tracking_counter:07d}{random.choice(letters)}"

def generate_deliveries(count, trip_ids):
    deliveries = []
    used_tracking = set()
    base_date = datetime(2025, 1, 1)
    for _ in range(count):
        trip_id = random.choice(trip_ids)
        tracking = generate_tracking_number()
        while tracking in used_tracking:
            tracking = generate_tracking_number()
        used_tracking.add(tracking)
        customer_name = fake.name()
        delivery_address = fake.address().replace('\n', ', ')
        package_weight = random.uniform(0.5, 50)
        scheduled = base_date + timedelta(days=random.randint(0, 365), hours=random.randint(8, 20))
        status = random.choice(DELIVERY_STATUSES)
        if status == 'delivered':
            delivered = scheduled + timedelta(hours=random.uniform(1, 8))
            signature = random.choice([True, False])
        else:
            delivered, signature = None, False
        deliveries.append((trip_id, tracking, customer_name, delivery_address, round(package_weight, 2), scheduled, delivered, status, signature))
    return deliveries

# Limpiar tabla antes de insertar para evitar UniqueViolation
cur.execute("TRUNCATE TABLE deliveries CASCADE")
print("Tabla deliveries limpiada")
print("Generando Deliveries (400,000 registros)...")
start_time = datetime.now()
deliveries_data = generate_deliveries(400000, trip_ids)
cur.executemany("INSERT INTO deliveries (trip_id, tracking_number, customer_name, delivery_address, package_weight_kg, scheduled_datetime, delivered_datetime, delivery_status, recipient_signature) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s) ON CONFLICT (tracking_number) DO NOTHING", deliveries_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(deliveries_data):,} entregas generadas en {duration:.2f}s")

Generando Deliveries (400,000 registros)...


UniqueViolation: llave duplicada viola restricción de unicidad «deliveries_tracking_number_key»
DETAIL:  Ya existe la llave (tracking_number)=(A3026156).


<div class="section">
<h3>6.6 🔧 Generación de Maintenance</h3>
</div>

In [ ]:
def generate_maintenance_records(count, vehicle_ids):
    records = []
    base_costs = {'Cambio de aceite': (150000, 300000), 'Revision de frenos': (200000, 500000), 'Alineacion y balanceo': (80000, 150000), 'Cambio de neumaticos': (400000, 800000), 'Revision de motor': (300000, 800000), 'Sistema electrico': (150000, 400000), 'Cambio de filtros': (100000, 250000), 'Inspeccion general': (50000, 150000), 'Mantenimiento preventivo': (200000, 500000)}
    for _ in range(count):
        vehicle_id = random.choice(vehicle_ids)
        maintenance_date = fake.date_between(start_date=datetime(2023, 1, 1), end_date=datetime(2025, 12, 31))
        maintenance_type = random.choice(MAINTENANCE_TYPES)
        description = f"{maintenance_type} - {fake.sentence()}"
        min_cost, max_cost = base_costs.get(maintenance_type, (100000, 500000))
        cost = random.uniform(min_cost, max_cost)
        next_date = maintenance_date + timedelta(days=random.randint(30, 180))
        performed_by = fake.name()
        records.append((vehicle_id, maintenance_date, maintenance_type, description, round(cost, 2), next_date, performed_by))
    return records

cur.execute("TRUNCATE TABLE maintenance CASCADE")
print("Tabla maintenance limpiada")
print("Generando Maintenance...")
start_time = datetime.now()
maintenance_data = generate_maintenance_records(5000, vehicle_ids)
cur.executemany("INSERT INTO maintenance (vehicle_id, maintenance_date, maintenance_type, description, cost, next_maintenance_date, performed_by) VALUES (%s, %s, %s, %s, %s, %s, %s)", maintenance_data)
duration = (datetime.now() - start_time).total_seconds()
print(f"{len(maintenance_data):,} registros de mantenimiento generados en {duration:.2f}s")

<div class="section">
<h2>📈 7. Resumen de Generación de Datos</h2>
</div>

In [ ]:
print("=" * 60)
print("RESUMEN DE GENERACION DE DATOS - FleetLogix Enterprise")
print("=" * 60)
cur.execute("SELECT 'vehicles' as tabla, COUNT(*) as registros FROM vehicles UNION ALL SELECT 'drivers', COUNT(*) FROM drivers UNION ALL SELECT 'routes', COUNT(*) FROM routes UNION ALL SELECT 'trips', COUNT(*) FROM trips UNION ALL SELECT 'deliveries', COUNT(*) FROM deliveries UNION ALL SELECT 'maintenance', COUNT(*) FROM maintenance")
results = cur.fetchall()
total = 0
print(f"\n{'Tabla':<20} {'Registros':>15}")
print("-" * 35)
for tabla, count in results:
    print(f"{tabla:<20} {count:>15,}")
    total += count
print("-" * 35)
print(f"{'TOTAL':<20} {total:>15,}")
print("=" * 60)

<div class="section">
<h2>📊 8. Visualización de Distribución de Datos</h2>
</div>

In [ ]:
df_vehicles = pd.read_sql("SELECT vehicle_type, COUNT(*) as count FROM vehicles GROUP BY vehicle_type", conn)
df_routes = pd.read_sql("SELECT destination_city, COUNT(*) as count FROM routes GROUP BY destination_city ORDER BY count DESC LIMIT 10", conn)
df_trips_status = pd.read_sql("SELECT status, COUNT(*) as count FROM trips GROUP BY status", conn)
df_deliveries_status = pd.read_sql("SELECT delivery_status, COUNT(*) as count FROM deliveries GROUP BY delivery_status", conn)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].bar(df_vehicles['vehicle_type'], df_vehicles['count'], color=sns.color_palette("husl", len(df_vehicles)))
axes[0, 0].set_title('Vehiculos por Tipo', fontsize=14, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 1].barh(df_routes['destination_city'], df_routes['count'], color=sns.color_palette("viridis", len(df_routes)))
axes[0, 1].set_title('Top 10 Rutas por Ciudad Destino', fontsize=14, fontweight='bold')
axes[1, 0].pie(df_trips_status['count'], labels=df_trips_status['status'], autopct='%1.1f%%', colors=sns.color_palette("Set2", len(df_trips_status)))
axes[1, 0].set_title('Distribucion de Viajes por Estado', fontsize=14, fontweight='bold')
axes[1, 1].pie(df_deliveries_status['count'], labels=df_deliveries_status['delivery_status'], autopct='%1.1f%%', colors=sns.color_palette("Set3", len(df_deliveries_status)))
axes[1, 1].set_title('Distribucion de Entregas por Estado', fontsize=14, fontweight='bold')
plt.suptitle('FleetLogix - Distribucion de Datos Generados', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<div class="section">
<h2>🔍 9. Control de Calidad de Datos (DQA)</h2>
</div>

In [ ]:
print("=" * 60)
print("VERIFICACIONES DE CALIDAD DE DATOS")
print("=" * 60)

checks = [
    ("Viajes sin vehiculo valido", "SELECT COUNT(*) FROM trips t LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id WHERE v.vehicle_id IS NULL"),
    ("Viajes sin conductor valido", "SELECT COUNT(*) FROM trips t LEFT JOIN drivers d ON t.driver_id = d.driver_id WHERE d.driver_id IS NULL"),
    ("Viajes sin ruta valida", "SELECT COUNT(*) FROM trips t LEFT JOIN routes r ON t.route_id = r.route_id WHERE r.route_id IS NULL"),
    ("Entregas sin viaje valido", "SELECT COUNT(*) FROM deliveries d LEFT JOIN trips t ON d.trip_id = t.trip_id WHERE t.trip_id IS NULL"),
    ("Inconsistencias temporales", "SELECT COUNT(*) FROM trips WHERE arrival_datetime IS NOT NULL AND arrival_datetime < departure_datetime"),
    ("Viajes con peso excedido", "SELECT COUNT(*) FROM trips t JOIN vehicles v ON t.vehicle_id = v.vehicle_id WHERE t.total_weight_kg > v.capacity_kg"),
    ("Vehiculos con campos nulos", "SELECT COUNT(*) FROM vehicles WHERE license_plate IS NULL OR vehicle_type IS NULL"),
]

all_pass = True
print(f"\n{'Verificacion':<40} {'Encontrados':>12} {'Estado':>12}")
print("-" * 65)
for name, query in checks:
    cur.execute(query)
    count = cur.fetchone()[0]
    passed = count == 0
    status = "PASS" if passed else "FAIL"
    print(f"{name:<40} {count:>12} {status:>12}")
    if not passed: all_pass = False
print("-" * 65)
print(f"\nRESULTADO FINAL: {'TODAS LAS VALIDACIONES PASS' if all_pass else 'HAY FALLOS'}")

<div class="section">
<h2>📈 10. Métricas Adicionales del Modelo</h2>
</div>

In [ ]:
print("=" * 60)
print("METRICAS ADICIONALES DEL MODELO")
print("=" * 60)
cur.execute("SELECT ROUND(AVG(delivery_count), 2) FROM (SELECT trip_id, COUNT(*) as delivery_count FROM deliveries GROUP BY trip_id) sub")
print(f"\nPromedio de entregas por viaje: {cur.fetchone()[0]}")
cur.execute("SELECT ROUND(AVG(trip_count), 2) FROM (SELECT vehicle_id, COUNT(*) as trip_count FROM trips GROUP BY vehicle_id) sub")
print(f"Promedio de viajes por vehiculo: {cur.fetchone()[0]}")
cur.execute("SELECT ROUND(AVG(fuel_consumed_liters), 2) FROM trips WHERE status = 'completed'")
print(f"Consumo promedio de combustible por viaje: {cur.fetchone()[0]} L")
cur.execute("SELECT ROUND(AVG(cost), 2) FROM maintenance")
print(f"Costo promedio de mantenimiento: ${cur.fetchone()[0]:,.2f}")
cur.execute("SELECT ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM deliveries), 2) FROM deliveries WHERE delivery_status = 'delivered'")
print(f"Tasa de entregas exitosas: {cur.fetchone()[0]}%")
print("=" * 60)

<div class="section">
<h2>🔚 11. Cierre de Conexión</h2>
</div>

In [ ]:
cur.close()
conn.close()
print("Conexion a PostgreSQL cerrada")
print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO - AVANCE 1")
print("=" * 60)
print("""
GENERACION DE DATOS COMPLETADA

Se han generado mas de 535,000 registros en las 6 tablas del modelo,
respetando todas las reglas de negocio e integridad referencial.

CUMPLIMIENTO DE REQUISITOS HENRY:
  Poblacion coherente de las 6 tablas (>500k registros)
  Script Python robusto con Faker y Pandas
  Analisis del modelo relacional y diagrama ER
  Validacion de integridad referencial
  Consistencia temporal verificada
  Distribucion estadistica realista
  Metricas de calidad de datos documentadas
""")
print("=" * 60)

<div class="section">
<hr>
<h3>Información del Notebook</h3>
<ul>
<li><strong>Autor:</strong> Dody Dueñas</li>
<li><strong>Proyecto:</strong> FleetLogix Enterprise - Henry M2</li>
<li><strong>Fecha:</strong> Abril 2026</li>
<li><strong>Versión:</strong> 1.0</li>
</ul>
<p><em>Documento generado como parte del Avance 1 del Proyecto Integrador M2 - Henry Data Science</em></p>
</div>

<!-- PROFESSIONAL_FOOTER_v1 -->
---

## 🏁 Conclusiones de este Avance

- ✔ Schema con 6 entidades normalizadas hasta 3FN
- ✔ Faker con locale es_CO para coherencia regional
- ✔ Inserción optimizada con batches de 5000 + transactions
- ✔ ~535.000 registros generados en menos de 5 minutos

## ➡ Siguiente paso

Continúa con **`Avance_2_SQLAnalysis.ipynb`** para profundizar en la siguiente etapa del proyecto.

---

<div align="center">

*FleetLogix · Proyecto 2 · Henry Data Science · 2026*  
📧 dodydurema67@gmail.com

</div>
